# Exercise 05 — continuous streaming + anomalies

**Goal:** simulate a realistic IoT data feed: continuous, irregular, occasionally noisy. This is the producer side of the canonical pattern "sensors push → backend aggregates → alerts fire".

**How to use this notebook:**

1. In a separate tab, open [`exercise_02_consume_basic.ipynb`]   (exercise_02_consume_basic.ipynb), run *Step 3* and then the    *Live streaming* cell — it will sit and wait for messages.
2. Run the streaming producer below. Events should appear live in    the consumer.
3. Stop the producer with the **■** button when you're done.

In [ ]:
from confluent_kafka import Producer
from datetime import datetime
import json, time, random

producer = Producer({
    'bootstrap.servers': 'redpanda:29092',
    'client.id':         'streaming-producer',
})

houses  = ['haus_a', 'haus_b', 'haus_c', 'haus_d']
sensors = {
    'strom':  {'unit': 'kWh',   'min': 0.5, 'max': 15.0},
    'wasser': {'unit': 'Liter', 'min': 0.1, 'max': 50.0},
}
print('Ready.')

## Step 1 — continuous stream (1 event every 0.5–2 s)

**What's interesting here:** the loop never ends. We let `time.sleep` create a realistic *event rate*. In a real sensor deployment each device would push on its own timer; here we simulate all of them from one process.

**Task — fill in the event dict.** Use the names `sensor`, `haus`, `wert`, `einheit`, `timestamp`.

In [ ]:
count = 0
print(f'{"Time":>10} | {"#":>4} | {"Topic":>6} | {"Key":>7} | {"Value":>8}')
print('-' * 50)

try:
    while True:
        house = random.choice(houses)
        topic = random.choice(list(sensors.keys()))
        cfg   = sensors[topic]
        value = round(random.uniform(cfg['min'], cfg['max']), 2)

        event = json.dumps({
            # TODO: fill in the event fields
            #   'sensor', 'haus', 'wert', 'einheit', 'timestamp'
        })

        producer.produce(topic, key=house.encode(), value=event.encode())
        producer.flush()

        count += 1
        ts = datetime.now().strftime('%H:%M:%S')
        print(f'{ts:>10} | {count:>4} | {topic:>6} | {house:>7} | {value:>8.2f}')
        time.sleep(random.uniform(0.5, 2.0))

except KeyboardInterrupt:
    print(f'\nStopped. Total events sent: {count}')

## Task A — burst mode

Send **50 events as fast as possible** (no `sleep`).

**What to observe:**

- The producer barely sweats — `produce()` only enqueues, the   background thread sends asynchronously.
- One `flush()` at the *end* is enough; you don't need one per event   (and shouldn't, for performance).
- The consumer lags only briefly: Kafka was designed for exactly this   kind of bursty traffic.

In [ ]:
# TODO: send 50 events as fast as possible, then producer.flush() once at the end


## Task B — anomaly simulation

Real sensor streams contain **outliers** — a defective sensor, a lightning strike, someone tampering with the meter. You can't ignore them: they corrupt averages and trigger false alarms unless flagged.

**Task:** send 30 events where ~20 % are anomalies (`wert` ≥ 3× the normal max). Add an `'anomaly': True` flag in the JSON.

**Why ship the flag along with the data?** Two reasons:

1. *Lineage* — downstream you can audit how many anomalies were in    the original stream.
2. *Self-describing data* — the consumer doesn't need to repeat the    threshold logic.

After running, find one of your anomalous messages in Redpanda Console (search the *Messages* tab for `"anomaly": true`).

In [ ]:
# TODO: send 30 events; ~20% should have wert > 3*max with 'anomaly': True
# Print '<-- ANOMALY' next to anomalous rows.


## What you learned

- Kafka producers are non-blocking — call `produce()` in a tight   loop and `flush()` only at the end.
- A *streaming* mindset: data is infinite, the program is the   pipeline, not the result.
- Including an explicit `anomaly` flag is cheaper and more reliable   than re-deriving it downstream.